# Run the article examples

Open this notebook from the extracted package directory. Install `requirements.txt` first. Run the cells in order. Outputs below were obtained by sequential Python execution; a separate Jupyter kernel was not tested. The appendix exercises missing coverage and reproduces the retrieval comparison.

# Knowledge graphs and GraphRAG

*Ontology 101, Part 2 of 5. Connect business records to the tickets and help pages that mention them.*

![Knowledge graphs and GraphRAG, Ontology 101 Part 2.](figures/00-cover.png)

Hooli has an active Enterprise subscription, and no product activity in the last thirty days. Vandelay also has no recent activity. Its subscription has lapsed. Wayne has stopped producing events. Its subscription remains active.

A customer success team wants to know which reported problems could explain the inactivity. The facts are in more than one place.

- subscription records
- usage events
- support tickets
- help documents

Selecting similar paragraphs is one way to begin. The team also needs to know which account each paragraph describes.

[Part 1](https://medium.com/towards-artificial-intelligence/what-an-ontology-actually-is-75fc2606c7e8) defined accounts, subscriptions, and plans. It also wrote a rule for an account with no active subscription. This part uses the same domain. It connects those records to evidence in text.

Here is what the lab returns on the packaged data. The contract rule lists accounts with no active subscription in a complete snapshot. That list is Initech and Vandelay. The usage rule lists accounts with zero events in a complete thirty-day window. That list is Hooli, Vandelay, and Wayne. The two lists overlap only at Vandelay. Active paid monthly recurring revenue on the full snapshot is $22,350. The same measure on the usage-inactive accounts is $6,200. That $6,200 is associated inactivity. It is not a forecast of lost revenue.

The same three accounts are the support question. A word-counting retrieve of the tickets and help documents is TF-IDF. It got 20 percent of the ten sources we marked as relevant. Filtering by account metadata got 70 percent. Following account metadata to tickets, then error codes to help documents, got 100 percent. The word counts of those three result sets are 170, 293, and 518. TF-IDF here is lexical retrieval. The lab does not evaluate a dense embedding model, so this result does not prove a general limit of vector search. You can also write the path as ordinary joins. The graph is not required for a small lookup.

The data is synthetic. The lab runs in Python. It does not need an API key. It does not need a database server or a language model. It builds the retrieval and evidence preparation stages of GraphRAG. GraphRAG means using a graph to select or organize evidence before a model writes an answer. The example answer later in this article is written from the records. It is not model output. An optional prompt in the package shows how to add a model later.

## Run the lab

Extract the companion ZIP. Open a terminal in its `ontology-101-part-2` directory. Use Python 3.11 or newer. Install the tested dependencies.

```bash
python -m pip install -r requirements.txt
python lab.py
```

`lab.py` runs the full example and writes results to `outputs/`. The notebook explains the same functions in order. You can also run the Python snippets in this article in order from the package directory.

## Last week's file and this week's graphs

Last week's ontology names the types and rules. This week's graph records particular accounts and subscriptions. The ontology can define `Account` and `Subscription`. The graph records that Acme has subscription `Sub_001`.

The three facts look like this in RDF. RDF is a W3C data model. Each fact has a subject, a predicate, and an object. Turtle is a readable syntax for those facts. The prefix shortens the shared part of the identifiers.

```turtle
@prefix saas: <http://example.org/saas#> .

saas:Acme saas:hasSubscription saas:Sub_001 .
saas:Sub_001 saas:onPlan saas:Enterprise .
saas:Sub_001 saas:isActive true .
```

The first statement connects an account to a subscription. The second connects that subscription to a plan. The third gives the subscription a boolean value. The repeated `Sub_001` identifier connects the statements into a graph. A triple is one statement. Several triples can describe the same entity. [W3C RDF primer](https://www.w3.org/TR/rdf11-primer/)

The ontology adds information about the vocabulary. It declares `Enterprise` to be a `PaidPlan`, and `PaidPlan` to be a subclass of `Plan`. A program can classify Enterprise as a plan by following that hierarchy.

![Acme's statements, a graph view, and the ontology declarations that describe their meaning.](figures/01-statements-and-model.svg)

*Read the relationships from left to right. The instance statements identify particular records. The ontology describes their types and rules.*

You also need identifiers to avoid accidental joins. Two systems may both have a record named `Pro`. The names alone do not establish that the records are the same plan. The lab keeps full IRIs as keys. IRI means international resource identifier. Short names are display labels. In production, a stable CRM identifier and an explicit mapping are safer than deriving identity from a changeable account name.

## Two graphs, two jobs

In a labeled property graph you store nodes and directed relationships, with properties attached to them. You can keep Acme, Sub_001, and Enterprise as nodes, then store `isActive` as a property of Sub_001. You can store the same domain in RDF or in a property graph.

Both models can hold a subscription's price and dates. The question is which operations you need. This lab uses NetworkX, an in-memory graph library, for traversals and community detection. It keeps the Turtle file for the ontology definitions. If you load either representation, you do not run an OWL reasoner.

The lab uses two working graphs.

| Graph | Example connection | What the connection tells us |
| --- | --- | --- |
| Operational graph | Acme has subscription Sub_001 | The structured snapshot records a business relationship. |
| Evidence graph | Vandelay and EXP-504 occur in T-1006 | The same source contains those entities. Read the source to understand the relationship. |

The graphs share account identifiers, so retrieval can combine status with the account's tickets. A text extractor cannot decide what counts as churn. The ontology cannot infer a ticket's meaning from its words alone.

The lab also checks that each subscription has exactly one plan relationship. That is a data check. It is not OWL inference. An OWL functional object property allows at most one distinct object value, but two names can still denote the same individual. A record with two plan identifiers is not always an OWL contradiction. SHACL is one standard way to write comparable checks over RDF. [OWL 2 primer](https://www.w3.org/TR/owl2-primer/), [SHACL](https://www.w3.org/TR/shacl/), [Neo4j constraints](https://neo4j.com/docs/cypher-manual/current/schema/constraints/)

## Load the ontology without losing the definitions

In [1]:
from pathlib import Path
from rdflib import Graph
from lab import rdf_to_lpg, SAAS

DATA = Path("data")
g = Graph().parse(DATA / "saas_ontology.ttl", format="turtle")
G, audit = rdf_to_lpg(g)
print(audit["rdf_triples"], audit["projected_instance_triples"])
print(G.number_of_nodes(), G.number_of_edges())

79 15
7 4


Fifteen instance triples become seven nodes and four relationships. The other sixty-four triples remain in the ontology file. Their declarations support the source model. Named subclass relationships also expand labels. A triple count is a count of statements in a representation. It is not a percentage of meaning preserved.

The working graph is a projection. It is a selected representation of the source data. It keeps full IRIs and literal datatype information, including separate entries for multiple values. This conversion covers the named individuals in the packaged dataset. It is not a general RDF conversion tool.

In [2]:
print(sorted(G.nodes[str(SAAS.Enterprise)]["labels"]))

['http://example.org/saas#PaidPlan', 'http://example.org/saas#Plan']


Enterprise has both labels because the loader follows the subclass hierarchy. It stores the result of that walk, so a taxonomy change requires rebuilding the projection. The loader has not evaluated the OWL definition of `Churned`.

Part 1's Initech example is why that distinction exists. Knowing that Initech has one inactive subscription does not establish that it has no other subscription. The final OWL example in Part 1 declares the subscription list complete and makes `isActive` functional. That lets the reasoner rule out an active subscription. Those declarations are in the packaged ontology. The calculations below use explicit Python rules over a declared complete snapshot instead.

## Define the activity window before counting events

The dataset extends Part 1 with eight accounts and daily activity from June 1 through August 28, 2026. The full history spans eighty-nine calendar days. We evaluate the last thirty complete days as of August 29.

The interval begins on July 30 and ends just before August 29. The lower bound is inclusive. The upper bound is exclusive. An event on July 30 counts. An event on August 29 does not.

In [3]:
from datetime import date, timedelta

as_of = date(2026, 8, 29)
start = as_of - timedelta(days=30)
example_days = [date(2026, 7, 29), date(2026, 7, 30),
                date(2026, 8, 28), date(2026, 8, 29)]
print([(day.isoformat(), start <= day < as_of) for day in example_days])

[('2026-07-29', False), ('2026-07-30', True), ('2026-08-28', True), ('2026-08-29', False)]


![The thirty-day activity window and the different treatment of zero events and incomplete coverage.](figures/02-window-and-coverage.svg)

*The date test decides which events count. The coverage declaration decides whether an absence of events can support a zero.*

Missing rows need a second check. The synthetic generator writes positive daily counts and omits zero counts. If the feed covers the whole window for an account, no rows in that window mean zero recorded activity. If part of the feed is missing, the same absence cannot support a zero.

The companion files include `usage_coverage.json`. That file states the period and accounts for which the synthetic feed is complete. The loader returns `None` for incomplete coverage and zero for complete coverage without events. Completeness is stated for this dataset. The loader does not infer it by inspecting an empty result.

In [4]:
import json
from lab import load_accounts, load_usage, validate_snapshot

load_accounts(G, DATA / "accounts.csv")
coverage = json.loads((DATA / "usage_coverage.json").read_text())
load_usage(G, DATA / "usage_events.csv", coverage)
validate_snapshot(G)

The loader excludes dates at or after the cutoff. It updates the last observed event only for positive counts. A row containing zero is not evidence of activity.

One boundary case is an account with no rows anywhere in the file. With complete coverage, its count is zero and its last observed event is unknown. Without complete coverage, both the count and the last observed event are unknown. The notebook includes the case so you can inspect both results.

This is the same completeness point as Part 1. `FILTER NOT EXISTS` can test whether a query finds an active subscription. Using the absence of a match as a business conclusion still needs a completeness assumption. OWL inference and a query over observed records make different claims about missing information. [SPARQL negation](https://www.w3.org/TR/sparql11-query/#negation)

## See where the two definitions disagree

The operational graph now contains ten accounts, ten subscriptions, and three plans. Every account has a subscription in the packaged data. Every subscription points to one plan.

The contract rule selects accounts with no active subscription in the complete snapshot. The usage rule selects accounts with zero events in the complete thirty-day window. These are the two lists from the opening. The code below is how you get them.

In [5]:
from lab import contract_inactive, usage_inactive, active_paid_mrr, label

contract = [label(a) for a in contract_inactive(G)]
usage = [label(a) for a in usage_inactive(G)]
print(contract)
print(usage)
print(active_paid_mrr(G))
print(active_paid_mrr(G, usage_inactive(G)))

['Initech', 'Vandelay']
['Hooli', 'Vandelay', 'Wayne']
22350.0
6200


![All ten accounts grouped by subscription status and activity in the complete window.](figures/03-account-matrix.svg)

*The matrix separates the definitions before you choose a metric name. The two inactive lists overlap only at Vandelay.*

Initech has no active subscription, and it records 1,298 events in the window. That situation may need a renewal or access-control investigation. Hooli and Wayne have active subscriptions and no recorded events. That suggests a different customer success action.

Hooli contributes $5,100 in active monthly recurring revenue. Wayne contributes $1,100. Their combined $6,200 is active paid MRR associated with inactivity. It is not a forecast that the company will lose that revenue.

The inactivity proportions are 2/10 and 3/10, or 20% and 30%. They are snapshot measures. They are not period churn rates. A period churn calculation needs an eligible starting cohort and dates of transition. The ten-account denominator also includes a Free account. You would need to handle that before naming either measure a paid-customer metric.

Unknown coverage must stay visible too. Removing an account with incomplete data from an inactivity list does not make that account active. A production report should show the unknown count. It should also state whether the proportion uses only observable accounts, or is withheld until coverage is complete.

## Turn a ticket into evidence you can check

The support corpus contains twenty tickets and six help documents. We use the ticket's account field to identify its owner. Then we match known entity names and error codes in the text.

Consider T-1006, a Vandelay ticket. Its body contains this sentence. "Scheduled Exports still fails with EXP-504 on anything full-size, and your support engineer told us the legacy export engine is the root cause." The ticket records a customer's attribution to a support engineer. It is evidence of what the customer reported. It is not an independently verified technical diagnosis.

The matching step finds `Scheduled Exports`, `EXP-504`, and `legacy export engine`. It also adds Vandelay from the account metadata. Each text match keeps its source ID and character span, so you can recover the matching words from the source.

In [6]:
from lab import load_corpus, build_evidence_graph, nodes_of

corpus = load_corpus(DATA)
EG, mentions = build_evidence_graph(
    corpus, [label(a) for a in nodes_of(G, "Account")]
)
by_id = {source["id"]: source for source in corpus}
for mention in mentions:
    if mention["source_id"] == "T-1006" and mention["name"] == "EXP-504":
        left, right = mention["span"]
        print(mention["method"], by_id["T-1006"]["text"][left:right])

regex EXP-504


The dictionary provides canonical names for a small set of features, plans, and components. The regular expression recognizes the error-code format in this dataset. Entity resolution means deciding which identifier a mention refers to. The lab handles only the simple matches in its vocabulary. A production extractor would also need aliases and ambiguous names.

Neither technique understands negation. A sentence such as "EXP-504 no longer occurs" still contains the error code. So we store an error mention. We do not assert an active incident. A model extractor would need the same distinction between extracted text and a verified domain fact.

## Understand what an edge weight means

The evidence graph connects entities that appear in the same source. The edge weight is a count of how many distinct sources contain the pair. It is not a count of repeated words inside one ticket.

Vandelay and EXP-504 occur together in T-1002 and T-1006, so their edge weight is two. The account link comes from metadata. The error-code link comes from text. The edge keeps both source IDs.

In [7]:
from lab import entity_key

account_key = str(SAAS.Vandelay)
error_key = entity_key("EXP-504", "ErrorCode")
print(EG[account_key][error_key])

{'weight': 2, 'sources': ['T-1002', 'T-1006']}


![Two source records support an edge of weight two between Vandelay and EXP-504.](figures/04-mentions-and-weights.svg)

*An edge answers which sources the entities share. You still need the source text to interpret their relationship.*

Co-occurrence loses information. A help document might mention both an old engine and its replacement. The graph connects them. The connection does not tell you which engine has a limitation and which one resolves it. The graph is useful for finding evidence. The document still holds the statement you must read.

A document that mentions several entities can also create many edges. Ten matched entities produce forty-five distinct pairs. Broad help documents may connect topics that are otherwise separate. Inspect the vocabulary and document boundaries before you treat a graph's groups as meaningful.

## Explain communities through the grouping calculation

A community method finds groups with more internal connection weight than a reference model would expect. Louvain repeatedly considers moving nodes between groups to improve a score called modularity. It then treats the groups as larger nodes and repeats. The method is a heuristic. It does not guarantee the globally best partition. [NetworkX Louvain documentation](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.louvain.louvain_communities.html)

You can inspect one term in the calculation using Vandelay and EXP-504. A node's strength is the sum of its incident edge weights. Vandelay has strength 5. EXP-504 has strength 25. The graph's total edge weight is 112. Under the standard weighted modularity reference model, the expected weight for the pair is 5 × 25 / (2 × 112), about 0.56. The observed weight of two is larger.

That pair contributes positively to placing the nodes together at the default resolution. The algorithm still evaluates the other connections too. A single pair does not determine an entire community. The notebook prints the inputs to the calculation so you can check them against the graph.

In [8]:
from lab import detect_communities

strength_a = EG.degree(account_key, weight="weight")
strength_b = EG.degree(error_key, weight="weight")
m = EG.size(weight="weight")
print(strength_a, strength_b, int(m))
print(round(strength_a * strength_b / (2 * m), 2))
communities = detect_communities(EG)

5 25 112
0.56


![The export and SSO communities in the executed evidence graph.](figures/05-communities.svg)

*The figure shows two groups from the four-community result. Filled nodes are accounts. Line width represents shared-source count.*

The export group contains Initech, Vandelay, and Wayne with EXP-504 and the export terms. Hooli appears with SSO, SAML, and SSO-301. SSO means single sign-on. SAML is a standard used to exchange authentication information between an identity provider and a service. Those terms appear together in the fixture's login-related records.

The groups are not OWL classes. No definition states that membership in the export group makes an account inactive. The extractor vocabulary, source boundaries, graph weights, resolution, and algorithm all affect membership. The implementation orders graph construction and fixes the random seed so the supplied run is repeatable.

Microsoft's original GraphRAG approach uses hierarchical communities and generated community reports. This lab uses Louvain without generated reports. Leiden, used in that GraphRAG approach, addresses limits in Louvain's community structure. The two algorithms should not be assumed to return the same partition. [GraphRAG paper](https://arxiv.org/abs/2404.16130), [Louvain and Leiden](https://arxiv.org/abs/1810.08473)

## Separate preparation from answering a question

GraphRAG uses a graph to retrieve or organize evidence before a model writes an answer. Preparation happens before a question arrives. It can include these steps.

- entity extraction
- identity linking
- community detection
- community reports

At question time, the system selects relevant evidence and fits it into the model's context. The model then writes an answer. A fuller implementation may call a model during extraction. It may also call a model for community reports and for the final answer. This teaching lab uses deterministic extraction and stops at evidence preparation.

Local and global retrieval serve different questions. A local question might ask which problems Vandelay reported. A global question might ask which product problems recur across the support corpus.

Microsoft's local search uses entity embeddings to identify candidates, then combines graph relationships with source text and other relevant records. Its global search works over community reports, produces intermediate answers, and combines selected points into a final response. [Local search](https://microsoft.github.io/graphrag/query/local_search/), [global search](https://microsoft.github.io/graphrag/query/global_search/)

The named-account question in this lab needs a simpler local path. We already know the account identifiers, so a metadata lookup can select the right tickets. Communities help you inspect recurring topics. They are not required for that lookup.

## Follow Vandelay's evidence path

First, select tickets whose account metadata identifies Vandelay. Second, find the error codes mentioned in those tickets. Third, use those codes to retrieve related help documents. The resulting packet includes the original ticket text and Vandelay's status from the operational graph.

![Vandelay links to its tickets, the tickets mention EXP-504, and the code connects to two help documents.](figures/06-evidence-path.svg)

*Each relationship has a stated basis. Sharing an error code with another account does not transfer that account's statements to Vandelay.*

T-1002 and T-1006 both mention EXP-504. D-01 explains the timeout and describes a workaround. D-02 describes the replacement engine's availability and the planned migration for Pro plans. Vandelay's subscription snapshot identifies its plan as Pro.

In [9]:
from lab import account_evidence

packet = account_evidence("Vandelay", corpus, EG, G)
print(packet["status"])
print([s["id"] for s in packet["tickets"]])
print([s["id"] for s in packet["documents"]])

{'plans': ['Pro'], 'subscription_active': False, 'events_30d': 0, 'last_observed_event': '2026-06-20'}
['T-1002', 'T-1006']
['D-01', 'D-02']


You get the account's reported symptoms and the documents needed to interpret its error code. The packet contains the source text, so a final answer can attach a citation to the claim it supports.

Initech's tickets remain evidence about Initech. They can support a statement that another account reported similar problems. They cannot establish what Vandelay experienced. Returning every source attached to a broad community would make that distinction harder to keep.

You could implement the same path with relational joins over account, ticket, mention, and document tables. A graph is helpful when you need to inspect those relationships, or reuse them across several traversals. It is not a requirement for a small lookup.

## Compare retrieval at the source level

The baseline uses TF-IDF. That name means term frequency and inverse document frequency. It gives weight to words based on their frequency in a source and how widely they occur across the corpus. The code normalizes the resulting vectors and ranks sources by cosine similarity to the query.

TF-IDF is lexical retrieval. A dense embedding model learns representations that can capture similarity beyond exact word overlap. The lab does not evaluate one. So the lexical results cannot establish a general limit of vector retrieval.

We compare the lexical baseline with a direct account metadata filter and the account-to-error path. Every method sees the same source collection. The lexical index includes account metadata. The question names Hooli, Vandelay, and Wayne.

The reference set contains seven account-owned tickets and three help documents explaining their errors. We identify those ten records by hand for this teaching question. Recall is the fraction of reference records retrieved. Precision is the fraction of returned records in that reference set.

![Source-level comparison of lexical retrieval, an account filter, and account plus error-code links.](figures/07-retrieval-coverage.svg)

*The methods retrieve different amounts of text. The result shows evidence coverage for one synthetic question. It is not a general performance ranking.*

TF-IDF's first four results include T-1012 and T-1016, plus two unrelated Data Import records. That is two of the ten reference sources, for 20% recall. Two of its four results are relevant, for 50% precision.

The account filter retrieves all seven account-owned tickets, reaching 70% recall. Following error-code links adds the three help documents and reaches 100% recall against the chosen reference set. The source text expands from 170 words to 293 words and then to 518 words. The counts exclude status metadata. They are whitespace-delimited words, not model tokens.

The larger evidence set has a cost. A fair performance experiment would compare methods under the same context budget and across additional questions. Keep the generation model constant, then measure supported claims, missing evidence, latency, and cost. The package records the exact source IDs and metric arithmetic so you can reproduce the one-question result.

For this small corpus, another baseline is to give a model all twenty-six records. A graph needs to justify its maintenance cost as the corpus and the questions become more demanding.

## Write an answer that matches the evidence

The retrieved sources support a qualified account-specific answer. The following is an editorial example written from the packaged evidence. It is not output from a model run.

> Vandelay reported repeated EXP-504 failures on Scheduled Exports. Its June 16 ticket says a support engineer attributed the problem to the legacy export engine and asks for a fix before a renewal decision. [T-1002, T-1006]
>
> The troubleshooting guide associates EXP-504 with the legacy engine's execution limit, and the release notes describe a later migration for Pro plans. Vandelay is on Pro in the subscription snapshot. [D-01, D-02, and subscription Sub_005]
>
> The usage file records no positive event after June 20 through the observation cutoff, and the snapshot shows an inactive subscription. The records support investigating export failures as a possible contributor to inactivity, but they do not include a cancellation date or a confirmed cancellation reason. [usage_events.csv and subscription Sub_005]

The answer separates the customer's report, the documentation, and the analyst's interpretation. Every citation has a specific job. An error-code guide can explain an error. It cannot establish which engine was deployed for an account on a particular date.

Hooli has a different set of reports. T-1010 describes a login loop after a SAML certificate rotation. T-1012 connects the continuing problem with declining adoption. T-1014 requests an administrator workaround. D-03 explains the certificate mismatch associated with SSO-301. The last observed positive event is July 15. That is consistent with the reports. It does not prove that SSO caused all later inactivity.

Wayne reports export problems in T-1013 and T-1016, then has no recorded events after July 28. Its subscription remains active at the snapshot date. The source evidence supports follow-up on the export issue. The current incident state and deployed engine remain unconfirmed.

Initech is the counterexample that blocks an overbroad explanation. It also reported EXP-504. It records 1,298 events in the thirty-day window. A shared error code can identify a problem worth investigating. It does not explain every account's behavior.

`generation_prompt.md` supplies an optional prompt for a model. It asks for the same separation between observations, reports, and hypotheses. Retrieving a correct source does not guarantee a supported answer. Generated claims would need their own evaluation.

## Choose the simplest method that answers the question

Use a direct lookup when the question identifies the account and the answer is in its records. Use text retrieval when the question describes a problem without a known entity or error code. Combine the two when finding relevant text requires both business identity and document content.

Community summaries can help with broad corpus questions, e.g. recurring product issues. The summaries need refresh and source attribution. An entity graph also needs maintenance when names, extraction rules, or source documents change.

For an enterprise implementation, apply the caller's access rules before building the evidence context. An accessible ticket must not become a route to an inaccessible document. Dated tickets also need a policy for resolved incidents and stale documentation. The lab's small corpus is a fixed synthetic snapshot. It does not implement authorization or tracking of incidents over time.

## Carry explicit definitions into Part 3

The lab writes these outputs.

- the operational graph
- the evidence graph
- source mentions
- complete account evidence packets

It also records dependency versions, source hashes, the activity window, and the comparison results.

The ontology remains the source of domain definitions. The working projection supports graph operations. The source records provide evidence. If you keep those roles explicit, you can inspect which assumptions produced an answer.

Part 3 will put the metric names, populations, owners, and time windows into a semantic layer. The same accounts will remain in the example. The next question will be which definition the organization intends when someone asks for churn.

## Exercise: no rows anywhere in the feed

An empty feed supports a zero only when its coverage is explicitly complete. We copy the graph so this exercise does not change the article results.

In [10]:
from copy import deepcopy
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmp:
    empty_feed = Path(tmp) / "empty.csv"
    empty_feed.write_text("account,date,events\n")
    known = deepcopy(G)
    load_usage(known, empty_feed, coverage)
    uncertain = deepcopy(G)
    partial_coverage = deepcopy(coverage)
    partial_coverage["complete_accounts"].remove("Hooli")
    load_usage(uncertain, empty_feed, partial_coverage)
    for name, graph in [("Complete", known), ("Incomplete", uncertain)]:
        record = graph.nodes[str(SAAS.Hooli)]
        print(name, record["events_30d"], record["last_observed_event"])
    assert known.nodes[str(SAAS.Hooli)]["events_30d"] == 0
    assert uncertain.nodes[str(SAAS.Hooli)]["events_30d"] is None

Complete 0 None
Incomplete None None


## Exercise: compare exact sources and evidence budgets

Inspect the returned IDs, the text volume, and the reference scores together. More retrieved text can improve coverage without proving that the method is better under equal budgets.

In [11]:
from lab import retrieval_comparison

comparison = retrieval_comparison(corpus, usage, EG, G)
for result in comparison:
    print(result["method"])
    print("Sources:", result["source_ids"])
    print("Words:", result["whitespace_words"],
          "Recall:", result["evidence_recall"],
          "Precision:", result["evidence_precision"])

TF-IDF top 4
Sources: ['T-1012', 'D-06', 'T-1016', 'T-1020']
Words: 170 Recall: 0.2 Precision: 0.5
Account metadata
Sources: ['T-1002', 'T-1006', 'T-1010', 'T-1012', 'T-1013', 'T-1014', 'T-1016']
Words: 293 Recall: 0.7 Precision: 1.0
Account + error links
Sources: ['D-01', 'D-02', 'D-03', 'T-1002', 'T-1006', 'T-1010', 'T-1012', 'T-1013', 'T-1014', 'T-1016']
Words: 518 Recall: 1.0 Precision: 1.0


## Export the walkthrough

Save the same graph, evidence packets, and metrics used by the article. Run `python test_lab.py` from a terminal for the regression checks.

In [12]:
from lab import save_outputs, json_ready

summary = save_outputs(G, EG, mentions, communities, corpus, audit, Path("outputs"))
print(json.dumps(json_ready(summary), indent=2))

{
  "active_paid_mrr": "22350.0",
  "contract_inactive": [
    "Initech",
    "Vandelay"
  ],
  "usage_inactive": [
    "Hooli",
    "Vandelay",
    "Wayne"
  ],
  "active_paid_mrr_on_inactive_accounts": "6200",
  "accounts": 10,
  "unknown_usage_accounts": []
}
